# 03b — Baseline LightGBM sin features de recencia (Fase 4)

Continuación de `03a_baseline_lightgbm.ipynb`, que detectó **leakage trivial**:
el target `churn_Nd = (user_days_since_last_login > N)` por construcción, y
esa misma feature está en la master. El árbol encontraba el split exacto en
`days > 14/30` con `best_iteration=1` y AUC=1.0000 perfecto.

## Solución aplicada
Lista negra de features de recencia post-hoc, excluida a nivel modelo. Las
masters quedan **intactas**. La regla:
- **Excluir**: features que miden "días desde la última vez que el jugador
  hizo X" (`*_days_since_*`, `*_days_ago` que se "congelan" al churn).
- **Mantener**: features de cohorte (`*_first_*`, `created_at_*` que son
  conocidas desde el día 1) y ventanas (`*_last_30d`, `*_last_balance`).

## Hallazgos del sondeo previo
- De 28 nombres en la lista negra del prompt, **13 existen en master original**
  (los otros 15 eran nombres tentativos que no se materializaron en Fase 1).
- 1 feature sospechosa no listada: `char_last_updated_days_ago` — el sanity
  check de AUC al final detectará si filtra.
- Otras features con `_last_` en el nombre son ventanas o estado, NO recencia.

## Outputs
- `data/data_models/model_<v>_<t>_no_recency.txt` (8 modelos)
- `data/data_models/predictions_<v>_<t>_no_recency.parquet` (8 archivos)
- `informes/fase3_modeling/03b_lightgbm_no_recency/baseline_comparison/` (métricas, gráficos, reports,
  `auc_sanity_check.csv` con flags de leakage residual)


In [1]:
# [SETUP] Imports
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, average_precision_score, log_loss, f1_score,
    confusion_matrix, roc_curve, precision_recall_curve,
)
from pathlib import Path
import json
import gc
import time
from datetime import datetime

assert lgb.__version__ >= '4.0', f'lightgbm >= 4.0 requerido'
print(f'lightgbm {lgb.__version__} OK')


lightgbm 4.6.0 OK


In [2]:
# [SETUP] Paths, constantes y lista negra
PROJECT_ROOT = Path.cwd().parents[1] if Path.cwd().name == 'fase3_modeling' else Path.cwd()
DATA_QC = PROJECT_ROOT / 'data' / 'data_qc'
DATA_MODELS = PROJECT_ROOT / 'data' / 'data_models'
REPORT_DIR = PROJECT_ROOT / 'informes' / 'fase3_modeling' / '03b_lightgbm_no_recency' / 'baseline_comparison'
DATA_MODELS.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
REFERENCE_DATE = pd.Timestamp('2026-04-04', tz='UTC')
CUTOFF_DATE = REFERENCE_DATE - pd.Timedelta(days=120)
# N_SAMPLE se lee dinámicamente del primer parquet (cell 4)
TARGETS = ['churn_14d', 'churn_30d']
MASTER_VERSIONS = {
    'original':         DATA_QC / 'master_table_cutoff.parquet',
    'v1_conservative':  DATA_QC / 'master_table_cutoff_v1_conservative.parquet',
    'v2_intermediate':  DATA_QC / 'master_table_cutoff_v2_intermediate.parquet',
    'v3_aggressive':    DATA_QC / 'master_table_cutoff_v3_aggressive.parquet',
}
SPLITS_PATH = DATA_MODELS / 'splits_indices_cutoff.parquet'

# N_SAMPLE leído dinámicamente del master original
N_SAMPLE = len(pd.read_parquet(MASTER_VERSIONS['original'], columns=['user_id']))

# Lista negra: features de recencia post-hoc al evento de churn
EXCLUDED_FEATURES = [
    # Bloque users — recencia
    'user_days_since_last_login',
    'user_last_login_date',
    'user_player_lifespan_days',
    # Bloque arena
    'arena_last_fight_days_ago', 'arena_days_since_last_fight',
    # Bloque char
    'char_last_action_days_ago', 'char_days_since_last_action',
    # Bloque coll
    'coll_days_since_last_item', 'coll_last_item_days_ago',
    # Bloque currency
    'tx_last_days_ago', 'tx_days_since_last',
    'gold_days_since_last_tx', 'gems_days_since_last_tx', 'dark_steel_days_since_last_tx',
    # Bloque devices
    'device_days_since_last_active', 'device_last_seen_days_ago',
    # Bloque feedback
    'feedback_days_since_last', 'feedback_last_days_ago',
    # Bloque fights
    'fights_last_days_ago', 'fights_days_since_last',
    # Bloque iap
    'iap_consumables_days_since_last', 'iap_days_since_subscription_end',
    'iap_subscription_days_since_last', 'iap_last_purchase_days_ago',
    # Bloque items
    'items_days_since_last_item', 'items_last_item_days_ago',
    # Bloque reward
    'reward_days_since_last_claim', 'reward_last_claim_days_ago',
]

# Validar inputs
for name, path in MASTER_VERSIONS.items():
    assert path.exists(), f'STOP: master {name} no encontrada en {path}'
assert SPLITS_PATH.exists(), f'STOP: splits no encontrados en {SPLITS_PATH}'

steps_log = []
NOTEBOOK_START = time.time()

def log_step(tag, message):
    ts = datetime.now().strftime('%H:%M:%S')
    entry = f'[{tag}] {ts} — {message}'
    steps_log.append(entry)
    print(entry)

print(f'PROJECT_ROOT: {PROJECT_ROOT}')
print(f'EXCLUDED_FEATURES: {len(EXCLUDED_FEATURES)} features en lista negra')
print(f'Masters validadas: {len(MASTER_VERSIONS)}')


PROJECT_ROOT: /Users/jezquerro/Documents/tfg
EXCLUDED_FEATURES: 28 features en lista negra
Masters validadas: 4


## BLOQUE 2 — Validación de lista negra y detección de sospechosas no listadas

In [3]:
# [ANALYSIS] Validar EXCLUDED_FEATURES contra cada master + detectar sospechosas

def is_cohort(col):
    """Heurística: ¿es feature de cohorte (mantener) en lugar de recencia (excluir)?"""
    return ('first' in col) or ('created_at' in col)


excluded_log = []
warnings_suspicious = {}

for vname, path in MASTER_VERSIONS.items():
    df = pd.read_parquet(path)
    cols = set(df.columns)

    found = [f for f in EXCLUDED_FEATURES if f in cols]
    not_found = [f for f in EXCLUDED_FEATURES if f not in cols]

    log_step('ANALYSIS', f'{vname}: {len(found)}/{len(EXCLUDED_FEATURES)} features de la lista negra encontradas')
    if not_found:
        log_step('INFO', f'{vname} — no presentes (ignorar): {not_found[:5]}{"..." if len(not_found)>5 else ""}')

    # Sospechosas: features con patrón temporal que NO están en lista negra ni son cohorte/target/key
    suspicious = []
    for c in df.columns:
        if c == 'user_id' or c in TARGETS:
            continue
        if c in EXCLUDED_FEATURES:
            continue
        if is_cohort(c):
            continue
        # Heurística estricta: termina en _days_ago O contiene days_since
        # (excluyendo *_last_<N>d, *_last_balance que son ventana/estado)
        is_temporal = (c.endswith('_days_ago') or 'days_since' in c)
        if is_temporal:
            suspicious.append(c)

    if suspicious:
        log_step('WARNING', f'{vname}: features sospechosas NO en lista negra: {suspicious}')
        warnings_suspicious[vname] = suspicious
    else:
        log_step('INFO', f'{vname}: sin sospechosas adicionales tras heurística')

    # Registrar para CSV de exclusión
    for f in found:
        excluded_log.append({
            'feature': f,
            'version': vname,
            'present': True,
        })
    for f in not_found:
        excluded_log.append({
            'feature': f,
            'version': vname,
            'present': False,
        })
    del df
    gc.collect()

# Generar excluded_features_list.csv (lista única + presencia por versión)
exc_pivot = pd.DataFrame(excluded_log).pivot_table(
    index='feature', columns='version', values='present', aggfunc='first', fill_value=False,
).astype(bool)

# Justificación por feature
JUST = {
    'user_days_since_last_login':       'CAUSA DIRECTA del leakage en 03a (=target)',
    'user_last_login_date':             'Misma información en formato timestamp',
    'user_player_lifespan_days':        'last_login - created → tautológico',
    'arena_last_fight_days_ago':        'recencia post-hoc',
    'arena_days_since_last_fight':      'recencia post-hoc',
    'char_last_action_days_ago':        'recencia post-hoc',
    'char_days_since_last_action':      'recencia post-hoc',
    'coll_days_since_last_item':        'recencia post-hoc',
    'coll_last_item_days_ago':          'recencia post-hoc',
    'tx_last_days_ago':                 'recencia post-hoc (transacciones)',
    'tx_days_since_last':               'recencia post-hoc',
    'gold_days_since_last_tx':          'recencia post-hoc',
    'gems_days_since_last_tx':          'recencia post-hoc',
    'dark_steel_days_since_last_tx':    'recencia post-hoc',
    'device_days_since_last_active':    'recencia post-hoc',
    'device_last_seen_days_ago':        'recencia post-hoc',
    'feedback_days_since_last':         'recencia post-hoc',
    'feedback_last_days_ago':           'recencia post-hoc',
    'fights_last_days_ago':             'recencia post-hoc',
    'fights_days_since_last':           'recencia post-hoc',
    'iap_consumables_days_since_last':  'recencia post-hoc (compras)',
    'iap_days_since_subscription_end':  'recencia post-hoc (subscripción)',
    'iap_subscription_days_since_last': 'recencia post-hoc',
    'iap_last_purchase_days_ago':       'recencia post-hoc',
    'items_days_since_last_item':       'recencia post-hoc',
    'items_last_item_days_ago':         'recencia post-hoc',
    'reward_days_since_last_claim':     'recencia post-hoc',
    'reward_last_claim_days_ago':       'recencia post-hoc',
}

exc_csv_rows = []
for f in EXCLUDED_FEATURES:
    versions_present = [v for v in MASTER_VERSIONS.keys() if (f in exc_pivot.index and bool(exc_pivot.loc[f, v]))]
    exc_csv_rows.append({
        'feature': f,
        'justification': JUST.get(f, '?'),
        'present_in_versions': ', '.join(versions_present) if versions_present else 'NONE',
    })
exc_csv_df = pd.DataFrame(exc_csv_rows)
exc_csv_df.to_csv(REPORT_DIR / 'excluded_features_list.csv', index=False)
print(f'\nexcluded_features_list.csv guardado ({len(exc_csv_df)} entradas)')
print(exc_csv_df.head(10).to_string(index=False))


[ANALYSIS] 19:23:49 — original: 10/28 features de la lista negra encontradas
[INFO] 19:23:49 — original — no presentes (ignorar): ['arena_last_fight_days_ago', 'arena_days_since_last_fight', 'char_last_action_days_ago', 'char_days_since_last_action', 'coll_last_item_days_ago']...
[WARNING] 19:23:49 — original: features sospechosas NO en lista negra: ['char_last_updated_days_ago', 'user_last_login_days_ago']
[ANALYSIS] 19:23:49 — v1_conservative: 8/28 features de la lista negra encontradas
[INFO] 19:23:49 — v1_conservative — no presentes (ignorar): ['user_days_since_last_login', 'user_last_login_date', 'arena_last_fight_days_ago', 'arena_days_since_last_fight', 'char_last_action_days_ago']...
[WARNING] 19:23:49 — v1_conservative: features sospechosas NO en lista negra: ['char_last_updated_days_ago']


[ANALYSIS] 19:23:49 — v2_intermediate: 5/28 features de la lista negra encontradas
[INFO] 19:23:49 — v2_intermediate — no presentes (ignorar): ['user_days_since_last_login', 'user_last_login_date', 'arena_last_fight_days_ago', 'arena_days_since_last_fight', 'char_last_action_days_ago']...
[WARNING] 19:23:49 — v2_intermediate: features sospechosas NO en lista negra: ['char_last_updated_days_ago']
[ANALYSIS] 19:23:49 — v3_aggressive: 4/28 features de la lista negra encontradas
[INFO] 19:23:49 — v3_aggressive — no presentes (ignorar): ['user_days_since_last_login', 'user_last_login_date', 'user_player_lifespan_days', 'arena_last_fight_days_ago', 'arena_days_since_last_fight']...
[WARNING] 19:23:49 — v3_aggressive: features sospechosas NO en lista negra: ['char_last_updated_days_ago']

excluded_features_list.csv guardado (28 entradas)
                    feature                              justification                                       present_in_versions
 user_days_since_last_login 

## BLOQUE 3 — Función de entrenamiento (idéntica a 03a + lista negra)

In [4]:
# [EXEC] Función de entrenamiento adaptada
def train_lightgbm_no_recency(master_path, target_col, version_name, target_name, splits_df, excluded_features):
    """Igual que 03a pero excluyendo EXCLUDED_FEATURES antes del entrenamiento."""
    t0 = time.time()

    df = pd.read_parquet(master_path).reset_index(drop=True)
    assert len(df) == N_SAMPLE

    # Separar X / y / aplicar lista negra
    target_cols_to_drop = ['churn_14d', 'churn_30d', 'user_id']
    excluded_present = [f for f in excluded_features if f in df.columns]

    y = df[target_col].astype(int)
    X = df.drop(columns=target_cols_to_drop + excluded_present)

    log_step('EXEC', f'[{version_name}/{target_name}] features: {X.shape[1]} (excluidas: {len(excluded_present)})')

    # Categóricas
    cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
    for c in cat_cols:
        X[c] = X[c].astype('category')

    # Splits
    train_mask = splits_df['split'].values == 'train'
    val_mask = splits_df['split'].values == 'val'
    test_mask = splits_df['split'].values == 'test'
    X_train, y_train = X[train_mask], y[train_mask]
    X_val, y_val = X[val_mask], y[val_mask]
    X_test, y_test = X[test_mask], y[test_mask]

    # Entrenar (mismos hiperparámetros que 03a)
    model = lgb.LGBMClassifier(
        objective='binary',
        metric='auc',
        n_estimators=500,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[lgb.early_stopping(50, verbose=False)],
        categorical_feature=cat_cols if cat_cols else 'auto',
    )

    # Predicciones y métricas
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_pred_proba >= 0.5).astype(int)

    auc_roc = float(roc_auc_score(y_test, y_pred_proba))
    auc_pr = float(average_precision_score(y_test, y_pred_proba))
    ll = float(log_loss(y_test, y_pred_proba))
    f1 = float(f1_score(y_test, y_pred))
    n_top = max(1, int(0.10 * len(y_test)))
    top_idx = np.argsort(y_pred_proba)[-n_top:]
    recall_at_10 = float(y_test.iloc[top_idx].sum()) / max(1, int(y_test.sum()))

    cm = confusion_matrix(y_test, y_pred)
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    precision_c, recall_c, _ = precision_recall_curve(y_test, y_pred_proba)

    fi_df = pd.DataFrame({
        'feature': X.columns,
        'importance_gain': model.booster_.feature_importance(importance_type='gain'),
        'importance_split': model.booster_.feature_importance(importance_type='split'),
    }).sort_values('importance_gain', ascending=False).reset_index(drop=True)

    elapsed = time.time() - t0
    log_step('EXEC', f'[{version_name}/{target_name}] AUC={auc_roc:.4f} | iter={model.best_iteration_} | t={elapsed:.1f}s')

    return {
        'version': version_name,
        'target': target_name,
        'n_features': X.shape[1],
        'n_categorical': len(cat_cols),
        'excluded_count': len(excluded_present),
        'excluded_features': excluded_present,
        'best_iteration': int(model.best_iteration_) if model.best_iteration_ else int(model.n_estimators_),
        'auc_roc': auc_roc,
        'auc_pr': auc_pr,
        'log_loss': ll,
        'f1': f1,
        'recall_at_10': recall_at_10,
        'confusion_matrix': cm,
        'fpr': fpr,
        'tpr': tpr,
        'precision_curve': precision_c,
        'recall_curve': recall_c,
        'feature_importance': fi_df,
        'model': model,
        'predictions': pd.DataFrame({
            'user_id': df.loc[test_mask, 'user_id'].values,
            'y_true': y_test.values,
            'y_pred_proba': y_pred_proba,
            'y_pred': y_pred,
        }),
        'elapsed_seconds': elapsed,
    }

print('Función definida')


Función definida


## BLOQUE 4 — Entrenar los 8 modelos

In [5]:
# [EXEC] Entrenar 8 modelos (4 versiones × 2 targets)
splits_df = pd.read_parquet(SPLITS_PATH)

results = {}
total_start = time.time()
for version_name, master_path in MASTER_VERSIONS.items():
    for target_col in TARGETS:
        key = f'{version_name}__{target_col}'
        log_step('EXEC', f'>>> {key}')
        results[key] = train_lightgbm_no_recency(
            master_path=master_path,
            target_col=target_col,
            version_name=version_name,
            target_name=target_col,
            splits_df=splits_df,
            excluded_features=EXCLUDED_FEATURES,
        )
        gc.collect()

total_elapsed = time.time() - total_start
print(f'\n=== 8 modelos entrenados en {total_elapsed:.1f}s ({total_elapsed/60:.1f} min) ===')
log_step('EXEC', f'8 modelos no_recency entrenados en {total_elapsed:.1f}s')


[EXEC] 19:23:49 — >>> original__churn_14d
[EXEC] 19:23:49 — [original/churn_14d] features: 102 (excluidas: 10)


[EXEC] 19:23:50 — [original/churn_14d] AUC=0.9283 | iter=1 | t=1.3s
[EXEC] 19:23:50 — >>> original__churn_30d
[EXEC] 19:23:50 — [original/churn_30d] features: 102 (excluidas: 10)


[EXEC] 19:23:52 — [original/churn_30d] AUC=0.9550 | iter=17 | t=1.6s
[EXEC] 19:23:52 — >>> v1_conservative__churn_14d
[EXEC] 19:23:52 — [v1_conservative/churn_14d] features: 85 (excluidas: 8)


[EXEC] 19:23:54 — [v1_conservative/churn_14d] AUC=0.8337 | iter=54 | t=1.9s
[EXEC] 19:23:54 — >>> v1_conservative__churn_30d
[EXEC] 19:23:54 — [v1_conservative/churn_30d] features: 85 (excluidas: 8)


[EXEC] 19:23:57 — [v1_conservative/churn_30d] AUC=0.7805 | iter=106 | t=2.9s
[EXEC] 19:23:57 — >>> v2_intermediate__churn_14d
[EXEC] 19:23:57 — [v2_intermediate/churn_14d] features: 83 (excluidas: 5)


[EXEC] 19:23:59 — [v2_intermediate/churn_14d] AUC=0.8337 | iter=54 | t=1.8s
[EXEC] 19:23:59 — >>> v2_intermediate__churn_30d
[EXEC] 19:23:59 — [v2_intermediate/churn_30d] features: 83 (excluidas: 5)


[EXEC] 19:24:01 — [v2_intermediate/churn_30d] AUC=0.7803 | iter=49 | t=1.8s
[EXEC] 19:24:01 — >>> v3_aggressive__churn_14d
[EXEC] 19:24:01 — [v3_aggressive/churn_14d] features: 73 (excluidas: 4)


[EXEC] 19:24:02 — [v3_aggressive/churn_14d] AUC=0.8322 | iter=38 | t=1.7s
[EXEC] 19:24:02 — >>> v3_aggressive__churn_30d
[EXEC] 19:24:02 — [v3_aggressive/churn_30d] features: 73 (excluidas: 4)


[EXEC] 19:24:04 — [v3_aggressive/churn_30d] AUC=0.7783 | iter=57 | t=2.0s

=== 8 modelos entrenados en 15.4s (0.3 min) ===
[EXEC] 19:24:04 — 8 modelos no_recency entrenados en 15.4s


## BLOQUE 5 — Sanity check de leakage residual

In [6]:
# [ANALYSIS] AUC sanity check + flags de leakage residual
sanity_data = []
for key, res in results.items():
    auc = res['auc_roc']
    fi = res['feature_importance']
    top_feature = fi.iloc[0]['feature']
    top_importance = fi.iloc[0]['importance_gain']
    second_importance = fi.iloc[1]['importance_gain'] if len(fi) > 1 else 1
    importance_ratio = top_importance / max(second_importance, 1e-9)

    flags = []
    if auc > 0.95:
        flags.append('AUC SOSPECHOSAMENTE ALTO (>0.95)')
    elif auc > 0.90:
        flags.append('AUC alto (>0.90), revisar')
    elif auc < 0.55:
        flags.append('AUC sub-baseline (<0.55)')
    if importance_ratio > 5:
        flags.append(f'feature top domina (ratio {importance_ratio:.1f}x)')
    flag_str = ' | '.join(flags) if flags else 'OK'

    sanity_data.append({
        'version': res['version'],
        'target': res['target'],
        'auc_roc': round(auc, 4),
        'best_iter': res['best_iteration'],
        'top_feature': top_feature,
        'top_importance': round(float(top_importance), 1),
        'importance_ratio_top_vs_second': round(importance_ratio, 2),
        'flag': flag_str,
    })

sanity_df = pd.DataFrame(sanity_data)
sanity_df.to_csv(REPORT_DIR / 'auc_sanity_check.csv', index=False)
print(sanity_df.to_string(index=False))

# Imprimir warnings explícitos para AUC > 0.95
for _, row in sanity_df.iterrows():
    if row['auc_roc'] > 0.95:
        log_step('WARNING', f'{row["version"]}/{row["target"]}: AUC={row["auc_roc"]} con top feature "{row["top_feature"]}". Posible leakage residual — añadir a lista negra y reentrenar.')

log_step('ANALYSIS', f'sanity check guardado: {len(sanity_df)} modelos')


        version    target  auc_roc  best_iter                top_feature  top_importance  importance_ratio_top_vs_second                                                                  flag
       original churn_14d   0.9283          1          user_game_version          6729.3                            2.60                                           AUC alto (>0.90), revisar
       original churn_30d   0.9550         17          user_game_version        109370.5                           23.71 AUC SOSPECHOSAMENTE ALTO (>0.95) | feature top domina (ratio 23.7x)
v1_conservative churn_14d   0.8337         54 char_last_updated_days_ago         51295.2                            3.14                                                                    OK
v1_conservative churn_30d   0.7805        106 char_last_updated_days_ago         35192.1                            2.54                                                                    OK
v2_intermediate churn_14d   0.8337         54 cha

## BLOQUE 6 — Guardado de modelos, predicciones y feature importance

In [7]:
# [EXEC] Guardado (sufijo _no_recency en nombres)
for key, res in results.items():
    version, target = key.split('__')

    model_path = DATA_MODELS / f'model_{version}_{target}_no_recency_cutoff.txt'
    res['model'].booster_.save_model(str(model_path))

    pred_path = DATA_MODELS / f'predictions_{version}_{target}_no_recency_cutoff.parquet'
    res['predictions'].to_parquet(pred_path, engine='pyarrow', index=False)

    fi_path = REPORT_DIR / f'feature_importance_{version}_{target}.csv'
    res['feature_importance'].to_csv(fi_path, index=False)

print(f'8 modelos + predicciones + feature_importance guardados')
log_step('EXEC', f'Outputs guardados en {DATA_MODELS}')


8 modelos + predicciones + feature_importance guardados
[EXEC] 19:24:05 — Outputs guardados en /Users/jezquerro/Documents/tfg/data/data_models


## BLOQUE 7 — Tabla comparativa de métricas

In [8]:
# [ANALYSIS] Tabla comparativa
rows = []
for key, res in results.items():
    rows.append({
        'version': res['version'],
        'target': res['target'],
        'n_features': res['n_features'],
        'n_excluded': res['excluded_count'],
        'best_iteration': res['best_iteration'],
        'auc_roc': round(res['auc_roc'], 4),
        'auc_pr': round(res['auc_pr'], 4),
        'log_loss': round(res['log_loss'], 4),
        'f1': round(res['f1'], 4),
        'recall_at_10': round(res['recall_at_10'], 4),
        'tn': int(res['confusion_matrix'][0, 0]),
        'fp': int(res['confusion_matrix'][0, 1]),
        'fn': int(res['confusion_matrix'][1, 0]),
        'tp': int(res['confusion_matrix'][1, 1]),
        'elapsed_s': round(res['elapsed_seconds'], 1),
    })
metrics_df = pd.DataFrame(rows)
metrics_df.to_csv(REPORT_DIR / 'metrics_comparison.csv', index=False)
print('=== metrics_comparison.csv guardado ===')
for target in TARGETS:
    print(f'\n--- {target} (orden desc por AUC-ROC) ---')
    sub = metrics_df[metrics_df['target'] == target].sort_values('auc_roc', ascending=False)
    print(sub.to_string(index=False))
log_step('ANALYSIS', f'metrics_comparison: {len(metrics_df)} filas')


=== metrics_comparison.csv guardado ===

--- churn_14d (orden desc por AUC-ROC) ---
        version    target  n_features  n_excluded  best_iteration  auc_roc  auc_pr  log_loss     f1  recall_at_10   tn  fp   fn  tp  elapsed_s
       original churn_14d         102          10               1   0.9283  0.8698    0.6105 0.0000        0.2974 2509   0 1271   0        1.3
v1_conservative churn_14d          85           8              54   0.8337  0.7924    0.4353 0.6825        0.2880 2361 148  536 735        1.9
v2_intermediate churn_14d          83           5              54   0.8337  0.7924    0.4353 0.6825        0.2880 2361 148  536 735        1.8
  v3_aggressive churn_14d          73           4              38   0.8322  0.7899    0.4439 0.6783        0.2872 2373 136  549 722        1.7

--- churn_30d (orden desc por AUC-ROC) ---
        version    target  n_features  n_excluded  best_iteration  auc_roc  auc_pr  log_loss     f1  recall_at_10   tn  fp  fn   tp  elapsed_s
       origina

## BLOQUE 8 — Visualizaciones

In [9]:
# [ANALYSIS] 4 visualizaciones (mismo patrón que 03a)
COLORS = {'original': '#2c3e50', 'v1_conservative': '#3498db',
          'v2_intermediate': '#e67e22', 'v3_aggressive': '#c0392b'}

# 1) ROC curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax_idx, target in enumerate(TARGETS):
    ax = axes[ax_idx]
    for version_name in MASTER_VERSIONS.keys():
        key = f'{version_name}__{target}'
        res = results[key]
        ax.plot(res['fpr'], res['tpr'],
                label=f"{version_name} (AUC={res['auc_roc']:.3f})",
                linewidth=2, color=COLORS[version_name])
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
    ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title(f'ROC — {target} (sin recencia)')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'roc_curves.png', dpi=120, bbox_inches='tight')
plt.close()

# 2) PR curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax_idx, target in enumerate(TARGETS):
    ax = axes[ax_idx]
    for version_name in MASTER_VERSIONS.keys():
        key = f'{version_name}__{target}'
        res = results[key]
        ax.plot(res['recall_curve'], res['precision_curve'],
                label=f"{version_name} (AP={res['auc_pr']:.3f})",
                linewidth=2, color=COLORS[version_name])
    baseline = float(results[f'original__{target}']['predictions']['y_true'].mean())
    ax.axhline(baseline, color='k', linestyle='--', alpha=0.3, label=f'Baseline ({baseline:.2f})')
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title(f'PR — {target} (sin recencia)')
    ax.legend(loc='lower left', fontsize=9)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'pr_curves.png', dpi=120, bbox_inches='tight')
plt.close()

# 3) Confusion matrices (8 subplots)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for col_idx, version_name in enumerate(MASTER_VERSIONS.keys()):
    for row_idx, target in enumerate(TARGETS):
        key = f'{version_name}__{target}'
        cm = results[key]['confusion_matrix']
        ax = axes[row_idx, col_idx]
        im = ax.imshow(cm, cmap='Blues')
        ax.set_title(f'{version_name}\n{target} (AUC={results[key]["auc_roc"]:.3f})', fontsize=10)
        ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
        ax.set_xticklabels(['No churn', 'Churn']); ax.set_yticklabels(['No churn', 'Churn'])
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True' if col_idx == 0 else '')
        max_val = cm.max()
        for i in range(2):
            for j in range(2):
                ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                        color='white' if cm[i,j] > max_val/2 else 'black', fontsize=10)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.close()

# 4) Top features heatmap
top_n = 30
all_top_features = set()
for key, res in results.items():
    all_top_features.update(res['feature_importance'].head(top_n)['feature'].tolist())
feature_list = sorted(all_top_features)
matrix = pd.DataFrame(index=feature_list, columns=list(results.keys()), dtype=float)
for key, res in results.items():
    fi = res['feature_importance'].copy()
    fi['rank'] = fi['importance_gain'].rank(ascending=False, method='min')
    fi_dict = dict(zip(fi['feature'], fi['rank']))
    for f in feature_list:
        matrix.loc[f, key] = fi_dict.get(f, np.nan)

fig, ax = plt.subplots(figsize=(12, max(8, len(feature_list) * 0.25)))
masked = np.ma.masked_invalid(matrix.values.astype(float))
im = ax.imshow(masked, cmap='RdYlGn_r', aspect='auto')
ax.set_xticks(range(len(matrix.columns)))
ax.set_xticklabels([c.replace('__', '\n') for c in matrix.columns], rotation=45, ha='right', fontsize=9)
ax.set_yticks(range(len(feature_list)))
ax.set_yticklabels(feature_list, fontsize=8)
plt.colorbar(im, label='Rank (1=más importante)', ax=ax)
ax.set_title(f'Top {top_n} features × 8 modelos (sin features de recencia)')
plt.tight_layout()
plt.savefig(REPORT_DIR / 'top_features_heatmap.png', dpi=120, bbox_inches='tight')
plt.close()

print(f'4 visualizaciones guardadas en {REPORT_DIR}')
log_step('ANALYSIS', '4 visualizaciones generadas')

# Robust top features (consenso)
consensus = {}
for key, res in results.items():
    for f in res['feature_importance'].head(top_n)['feature'].tolist():
        consensus[f] = consensus.get(f, 0) + 1
robust = sorted([(f, n) for f, n in consensus.items() if n >= 6], key=lambda x: -x[1])
robust_df = pd.DataFrame([{'feature': f, 'n_models_top30': n} for f, n in robust])
robust_df.to_csv(REPORT_DIR / 'robust_top_features.csv', index=False)
print(f'robust_top_features.csv: {len(robust_df)} features en top {top_n} de >=6 modelos')


4 visualizaciones guardadas en /Users/jezquerro/Documents/tfg/informes/fase3_modeling/03b_lightgbm_no_recency/baseline_comparison
[ANALYSIS] 19:24:05 — 4 visualizaciones generadas
robust_top_features.csv: 25 features en top 30 de >=6 modelos


## BLOQUE 9 — Informes

In [10]:
# [REPORT] baseline_summary.md
now_str = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

# Determinar ganadores
winners = {}
for target in TARGETS:
    sub = metrics_df[metrics_df['target'] == target].sort_values('auc_roc', ascending=False)
    winners[target] = sub.iloc[0]['version']
    winners[f'{target}_delta'] = sub['auc_roc'].max() - sub['auc_roc'].min()

# Diff churn_14d vs churn_30d
auc_14d_best = metrics_df[metrics_df['target'] == 'churn_14d']['auc_roc'].max()
auc_30d_best = metrics_df[metrics_df['target'] == 'churn_30d']['auc_roc'].max()

# Detectar warnings de leakage residual
leakage_warnings = sanity_df[sanity_df['auc_roc'] > 0.95]

lines = [
    '# Resumen ejecutivo — Baseline LightGBM (sin features de recencia)',
    '',
    f'**Fecha**: {now_str}',
    f'**Tiempo total**: {sum(r["elapsed_seconds"] for r in results.values()):.1f}s',
    '',
    '## Contexto',
    '',
    '- Continuación de `03a_baseline_lightgbm.ipynb`, que detectó leakage trivial',
    '  por `user_days_since_last_login` (=target por construcción).',
    f'- Solución: lista negra de **{len(EXCLUDED_FEATURES)} features de recencia**, excluida a nivel modelo.',
    '- Las masters NO se modificaron — exclusión solo durante el entrenamiento.',
    '',
    '## Setup',
    '',
    '- 8 modelos: 4 versiones × 2 targets',
    '- Splits compartidos con 03a (mismos jugadores en train/val/test)',
    '- LightGBM defaults, early stopping 50 rondas, threshold 0.5',
    '',
    '## Resultados — churn_30d',
    '',
    '| Versión | n_features | AUC-ROC | AUC-PR | Recall@10% | F1 | Tiempo |',
    '|---|---|---|---|---|---|---|',
]
for v in sorted(MASTER_VERSIONS.keys(), key=lambda x: -metrics_df[(metrics_df['version']==x) & (metrics_df['target']=='churn_30d')]['auc_roc'].iloc[0]):
    sub = metrics_df[(metrics_df['version'] == v) & (metrics_df['target'] == 'churn_30d')].iloc[0]
    lines.append(f'| {v} | {sub["n_features"]} | {sub["auc_roc"]:.4f} | {sub["auc_pr"]:.4f} | {sub["recall_at_10"]:.4f} | {sub["f1"]:.4f} | {sub["elapsed_s"]}s |')

lines += ['', '## Resultados — churn_14d', '',
          '| Versión | n_features | AUC-ROC | AUC-PR | Recall@10% | F1 | Tiempo |',
          '|---|---|---|---|---|---|---|']
for v in sorted(MASTER_VERSIONS.keys(), key=lambda x: -metrics_df[(metrics_df['version']==x) & (metrics_df['target']=='churn_14d')]['auc_roc'].iloc[0]):
    sub = metrics_df[(metrics_df['version'] == v) & (metrics_df['target'] == 'churn_14d')].iloc[0]
    lines.append(f'| {v} | {sub["n_features"]} | {sub["auc_roc"]:.4f} | {sub["auc_pr"]:.4f} | {sub["recall_at_10"]:.4f} | {sub["f1"]:.4f} | {sub["elapsed_s"]}s |')

lines += ['', '## Sanity check de leakage residual', '',
          '| Versión | Target | AUC | best_iter | Top feature | Importance ratio | Flag |',
          '|---|---|---|---|---|---|---|']
for _, r in sanity_df.iterrows():
    lines.append(f'| {r["version"]} | {r["target"]} | {r["auc_roc"]} | {r["best_iter"]} | `{r["top_feature"]}` | {r["importance_ratio_top_vs_second"]} | {r["flag"]} |')

lines += ['', '## Hallazgos clave', '',
          '### ¿Qué versión de master gana?',
          '',
          f'- **churn_30d**: gana `{winners["churn_30d"]}` (delta={winners["churn_30d_delta"]:.4f})',
          f'- **churn_14d**: gana `{winners["churn_14d"]}` (delta={winners["churn_14d_delta"]:.4f})',
          '']

for target in TARGETS:
    delta = winners[f'{target}_delta']
    if delta < 0.005:
        verdict = 'prácticamente equivalentes — quedarse con v3 (más ligera)'
    elif delta < 0.02:
        verdict = 'diferencia leve — evaluar coste/beneficio'
    else:
        verdict = 'diferencia material — investigar features eliminadas'
    lines.append(f'- {target}: {verdict}')

lines += ['',
          '### ¿Qué features son consistentemente importantes?',
          '',
          f'Features en top 30 de >=6 de los 8 modelos (`robust_top_features.csv`):',
          '',
          '| Feature | Modelos en top 30 |',
          '|---|---|']
for f, n in robust[:15]:
    lines.append(f'| `{f}` | {n}/8 |')
if len(robust) > 15:
    lines.append(f'| ... | ({len(robust) - 15} más) |')

lines += ['', '### ¿Es razonable el rango de AUC?', '',
          f'- AUC máxima churn_14d: **{auc_14d_best:.4f}**',
          f'- AUC máxima churn_30d: **{auc_30d_best:.4f}**',
          '',
          'Literatura de referencia para churn en F2P sin cutoff temporal estricto:',
          '- Hadiji et al. (2014): AUC 0.74-0.83',
          '- Periáñez et al. (2016): AUC 0.78-0.86',
          '']

if 0.70 <= auc_14d_best <= 0.95 and 0.70 <= auc_30d_best <= 0.95:
    lines.append('Resultado **dentro del rango razonable** (0.70-0.95). Coherente con literatura.')
elif auc_14d_best > 0.95 or auc_30d_best > 0.95:
    lines.append('Resultado **sospechosamente alto** (>0.95). Probable leakage residual — revisar sanity_check.')
else:
    lines.append('Resultado **por debajo del rango esperado**. Investigar si features útiles se excluyeron por error.')

if not leakage_warnings.empty:
    lines += ['', '### Sospechas de leakage residual', '',
              f'{len(leakage_warnings)} modelos con AUC > 0.95. Features candidatas a añadir a la lista negra:']
    for _, r in leakage_warnings.iterrows():
        lines.append(f'- `{r["top_feature"]}` (en {r["version"]}/{r["target"]}, AUC={r["auc_roc"]})')

lines += ['', '## Próximos pasos', '',
          '1. Si AUC > 0.95 en alguna versión: ampliar lista negra y reentrenar (este notebook es idempotente).',
          '2. Tuning de hiperparámetros con Optuna sobre la versión ganadora.',
          '3. Otros algoritmos (XGBoost, LR) como control.',
          '4. SHAP para interpretabilidad.',
          '5. Análisis de errores: subgrupos donde el modelo falla.',
          '6. (Opcional, futuro) Reescribir Fase 1 con cutoff temporal explícito para recuperar señal de recencia legítima.',
          '']

SUMMARY_PATH = REPORT_DIR / 'baseline_summary.md'
with open(SUMMARY_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print(f'baseline_summary.md guardado: {SUMMARY_PATH}')
log_step('REPORT', 'baseline_summary.md generado')


baseline_summary.md guardado: /Users/jezquerro/Documents/tfg/informes/fase3_modeling/03b_lightgbm_no_recency/baseline_comparison/baseline_summary.md
[REPORT] 19:24:05 — baseline_summary.md generado


In [11]:
# [REPORT] execution_report.md
elapsed = time.time() - NOTEBOOK_START
t_min = int(elapsed // 60); t_sec = int(elapsed % 60)

lines = [
    '# Execution Report — 03b_baseline_lightgbm_no_recency',
    '',
    f'**Notebook**: notebooks/fase3_modeling/03b_baseline_lightgbm_no_recency.ipynb',
    f'**Fecha**: {now_str}',
    f'**Tiempo de ejecución**: {t_min} min {t_sec} s',
    f'**Modelos entrenados**: 8 (4 versiones × 2 targets)',
    f'**Features excluidas (lista negra)**: {len(EXCLUDED_FEATURES)} totales, {{count}} presentes en master original',
    '',
    '## Setup',
    '',
    f'- LightGBM {lgb.__version__}',
    f'- random_state = {RANDOM_STATE}',
    f'- Splits compartidos con 03a (`splits_indices_cutoff.parquet`)',
    f'- Hiperparámetros idénticos a 03a (defaults razonables)',
    '',
    '## Decisión metodológica',
    '',
    'En 03a se detectó leakage trivial: `churn_Nd = (user_days_since_last_login > N)`',
    'por construcción del target. AUC=1.0000 con best_iteration=1 confirmó que el',
    'modelo encontraba el split exacto en una sola feature.',
    '',
    'Solución elegida: **exclusión a nivel modelo** (este notebook). Las masters',
    'NO se modificaron porque arreglarlas requeriría:',
    '1. Reescribir 02a con cutoff temporal explícito (separar período de observación',
    '   de período de target).',
    '2. Re-ejecutar todos los notebooks de Fase 1.',
    '',
    'Eso queda como TODO para una iteración posterior. Por ahora, baseline conservador',
    'sin features de recencia post-hoc.',
    '',
    '## Lista negra aplicada',
    '',
    '| Categoría | Patrón | Razón de exclusión |',
    '|---|---|---|',
    '| Recencia general | `*_days_since_*`, `*_last_*_days_ago` | Se "congela" al churn, correlación tautológica con target |',
    '| Lifespan | `user_player_lifespan_days` | = `last_login - created`, tautológico |',
    '| Last login (variantes) | `user_last_login_date`, `user_days_since_last_login` | Causa directa del leakage |',
    '',
    'Features que SE MANTIENEN aunque parezcan candidatas:',
    '- `*_first_*` (cohorte: conocido día 1)',
    '- `user_created_at_days_ago` (cohorte)',
    '- `*_last_30d`, `*_last_balance` (ventana / estado, no recencia)',
    '',
    f'Lista completa documentada en `excluded_features_list.csv` ({len(EXCLUDED_FEATURES)} entradas).',
    '',
    '## Sospechosas detectadas no listadas',
    '',
]
if warnings_suspicious:
    for vname, susp in warnings_suspicious.items():
        lines.append(f'- **{vname}**: {susp}')
else:
    lines.append('Ninguna feature sospechosa adicional detectada por la heurística (`endswith _days_ago` o `contains days_since`).')

lines += ['',
          'El sanity check de AUC al final del notebook detecta features residuales',
          'que filtren con `top_feature` dominante (ratio importance > 5x sobre la segunda).',
          '',
          '## Resultados (las 8 entradas)',
          '',
          '| version | target | n_features | n_excluded | best_iter | AUC-ROC | AUC-PR | F1 | Recall@10 |',
          '|---|---|---|---|---|---|---|---|---|']
for r in metrics_df.sort_values(['target', 'auc_roc'], ascending=[True, False]).to_dict('records'):
    lines.append(f'| {r["version"]} | {r["target"]} | {r["n_features"]} | {r["n_excluded"]} | {r["best_iteration"]} | {r["auc_roc"]:.4f} | {r["auc_pr"]:.4f} | {r["f1"]:.4f} | {r["recall_at_10"]:.4f} |')

lines += ['',
          '## Sanity check de leakage residual',
          '',
          '| Versión | Target | AUC | best_iter | Top feature | Imp. ratio | Flag |',
          '|---|---|---|---|---|---|---|']
for _, r in sanity_df.iterrows():
    lines.append(f'| {r["version"]} | {r["target"]} | {r["auc_roc"]} | {r["best_iter"]} | `{r["top_feature"]}` | {r["importance_ratio_top_vs_second"]} | {r["flag"]} |')

lines += ['',
          '## Decisiones tomadas',
          '',
          '1. **Exclusión a nivel modelo**, no a nivel datos. Las masters quedan intactas.',
          f'2. **Lista negra explícita** ({len(EXCLUDED_FEATURES)} features). Las que no existen en una versión se ignoran silenciosamente.',
          '3. **Heurística de cohorte vs recencia**: `*_first_*` y `*_created_at_*` SE MANTIENEN.',
          '4. **Sanity check post-entrenamiento**: si AUC > 0.95, flag explícito + warning con feature candidata.',
          '5. **Sin tuning ni búsqueda de threshold** — comparable directamente con 03a.',
          '',
          '## Limitaciones',
          '',
          '- Sin features de recencia, el modelo pierde la señal más informativa de churn (recencia ES la señal real, lo que no podemos usar es la versión post-hoc del target).',
          '- El cutoff temporal correcto requiere re-arquitecturar Fase 1 (TODO).',
          '- Hiperparámetros sin tunear → AUC alcanzable es probablemente +0.02-0.05 con Optuna.',
          '',
          '## Pasos ejecutados',
          '']
for s in steps_log:
    lines.append(f'- {s}')

lines += ['',
          '## TODOs',
          '',
          '- [ ] Si sanity_check detecta AUC > 0.95: ampliar lista negra (likely candidate: `char_last_updated_days_ago`)',
          '- [ ] Tuning de hiperparámetros con Optuna sobre versión ganadora',
          '- [ ] Comparativa con XGBoost y LR como controles',
          '- [ ] SHAP para interpretabilidad por instancia',
          '- [ ] Análisis de errores en subgrupos (por país, por payer/no-payer)',
          '- [ ] Calibración de probabilidades (Platt / isotonic)',
          '- [ ] Búsqueda de threshold óptimo por target',
          '- [ ] (Largo plazo) Re-arquitectura de Fase 1 con cutoff temporal',
          '',
          '## Archivos generados',
          '',
          '| Archivo | Tipo |',
          '|---|---|',
          '| `data/data_models/model_<v>_<t>_no_recency.txt` | 8 modelos LightGBM |',
          '| `data/data_models/predictions_<v>_<t>_no_recency.parquet` | 8 archivos predicciones |',
          '| `informes/.../excluded_features_list.csv` | lista negra documentada |',
          '| `informes/.../metrics_comparison.csv` | tabla comparativa |',
          '| `informes/.../auc_sanity_check.csv` | sanity con flags |',
          '| `informes/.../feature_importance_<v>_<t>.csv` | 8 feature importances |',
          '| `informes/.../robust_top_features.csv` | features en top 30 de >=6 modelos |',
          '| `informes/.../roc_curves.png`, `pr_curves.png`, `confusion_matrices.png`, `top_features_heatmap.png` | visualizaciones |',
          '| `informes/.../baseline_summary.md`, `execution_report.md` | informes |',
          '']

# Reemplazar placeholder
n_excluded_orig = sum(1 for f in EXCLUDED_FEATURES if f in pd.read_parquet(MASTER_VERSIONS['original']).columns)
text = '\n'.join(lines).replace('{count}', str(n_excluded_orig))

REPORT_PATH = REPORT_DIR / 'execution_report.md'
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write(text)
print(f'execution_report.md guardado: {REPORT_PATH}')
log_step('REPORT', 'execution_report.md generado')


execution_report.md guardado: /Users/jezquerro/Documents/tfg/informes/fase3_modeling/03b_lightgbm_no_recency/baseline_comparison/execution_report.md
[REPORT] 19:24:06 — execution_report.md generado


In [12]:
# [REPORT] Resumen final en consola
total = time.time() - NOTEBOOK_START
print('=' * 70)
print(f'RESUMEN FINAL — Notebook 03b_baseline_lightgbm_no_recency')
print('=' * 70)
print(f'  Tiempo total                : {int(total//60)}m {int(total%60)}s')
print(f'  Modelos entrenados          : 8')
print(f'  Features excluidas          : {len(EXCLUDED_FEATURES)} en lista negra')
print()
print(f'  AUC-ROC máxima por target:')
for target in TARGETS:
    sub = metrics_df[metrics_df['target'] == target]
    best = sub.loc[sub['auc_roc'].idxmax()]
    print(f'    {target:<12s}: {best["auc_roc"]:.4f}  ({best["version"]})')
print()
print(f'  Delta AUC max-min:')
for target in TARGETS:
    sub = metrics_df[metrics_df['target'] == target]
    delta = sub['auc_roc'].max() - sub['auc_roc'].min()
    print(f'    {target:<12s}: {delta:.4f}')
print()
flagged = sanity_df[sanity_df['auc_roc'] > 0.95]
if not flagged.empty:
    print(f'  {len(flagged)} modelos con AUC > 0.95 (posible leakage residual):')
    for _, r in flagged.iterrows():
        print(f'    {r["version"]}/{r["target"]}: top={r["top_feature"]}')
else:
    print(f'  Sanity check OK: ningún modelo con AUC > 0.95')
print()
print(f'  Outputs:')
print(f'    data/data_models/  (8 modelos + 8 predicciones)')
print(f'    {REPORT_DIR.relative_to(PROJECT_ROOT)}/')
print('=' * 70)


RESUMEN FINAL — Notebook 03b_baseline_lightgbm_no_recency
  Tiempo total                : 0m 16s
  Modelos entrenados          : 8
  Features excluidas          : 28 en lista negra

  AUC-ROC máxima por target:
    churn_14d   : 0.9283  (original)
    churn_30d   : 0.9550  (original)

  Delta AUC max-min:
    churn_14d   : 0.0961
    churn_30d   : 0.1767

  1 modelos con AUC > 0.95 (posible leakage residual):
    original/churn_30d: top=user_game_version

  Outputs:
    data/data_models/  (8 modelos + 8 predicciones)
    informes/fase3_modeling/03b_lightgbm_no_recency/baseline_comparison/


In [13]:
# [REPORT] Logging dual
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'scripts'))
from notebook_logging_template import (
    export_notebook_to_html, build_enriched_report_section,
)

notebook_path = PROJECT_ROOT / 'notebooks' / 'fase3_modeling' / '03b_baseline_lightgbm_no_recency.ipynb'
html_path = REPORT_DIR / '03b_baseline_lightgbm_no_recency_run.html'
export_notebook_to_html(notebook_path, html_path)

enriched = build_enriched_report_section(
    df_final=metrics_df,
    raw_shape=None,
    filter_steps=[
        ('Modelos planeados', 8),
        ('Modelos entrenados', len(results)),
        (f'Features excluidas (lista negra)', len(EXCLUDED_FEATURES)),
    ],
)
with open(REPORT_DIR / 'execution_report.md', 'a', encoding='utf-8') as f:
    f.write('\n\n---\n\n## Outputs detallados (metrics_df)\n\n' + enriched)
print(f'HTML + enriquecimiento appendeado')


HTML generado: 03b_baseline_lightgbm_no_recency_run.html (0.5 MB)
HTML + enriquecimiento appendeado
